In [ ]:
import os
import json

import dask.dataframe as dd
import geopandas as gpd
import numpy as np
import pandas as pd

from sklearn.metrics import root_mean_squared_error

### Functions

In [ ]:
def getFiles(filepath):
    files = []
    for file in os.listdir(filepath):
        if not file.startswith('.'):
            # if not 'C2' in file:
            files.append(os.path.join(filepath, file))
    return files

In [ ]:
data_dir = '/nas/cee-water/cjgleason/fiona/for_manuscripts/26_narrow_rivers/data/'
save_dir = '/nas/cee-water/cjgleason/fiona/for_manuscripts/26_narrow_rivers/results/'

### Setup

In [ ]:
# Order of all possible bins for plotting
order = ['(0, 10]', '(10, 20]', '(20, 30]', '(30, 40]', '(40, 50]',
         '(50, 60]', '(60, 70]', '(70, 80]', '(80, 90]', '(90, 100]',
         '(100, 150]', '(150, 200]', '(200, 500]', '(500, 1000]',
         '(1000, 5000]']

# Cutpoints
bins = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 150, 200, 500, 1000, 5000]

### My data

In [ ]:
data_dir = '/nas/cee-water/cjgleason/fiona/for_manuscripts/26_narrow_rivers/data/'
output_dir = '/nas/cee-water/cjgleason/fiona/narrow_rivers_PIXC_output/PIXC_vD_wse_2026_06_01_rel_abs/'

# Get list of files
files = getFiles(output_dir)
print(len(files))

# Read in all files
ddf = dd.read_parquet(files)

# Bin reaches by width for pre-filtering analysis
ddf['Bin'] = ddf['WidthM'].map_partitions(pd.cut, bins=bins).astype(str)
ddf = ddf.reset_index(drop=True)

# Count number of reaches before filtering
unfiltered_task = ddf.groupby('Bin', observed=False)['site_id'].nunique()
unfiltered = unfiltered_task.compute().reset_index(name='count_unfil')

unfiltered['Bin'] = pd.Categorical(unfiltered['Bin'], categories=order, ordered=True)
unfiltered = unfiltered.sort_values('Bin')

max_w = ddf.WidthM.max().compute()

# Sort data
ddf = ddf[
    ddf['wse_gauge'].notnull() & 
    ddf['wse_swot'].notnull()
    ][[
        'date', 'n_pixels', 'wse_gauge',
        'wse_swot', 'site_id', 'WidthM',
        'Bin'
    ]]

df = ddf.compute().sort_values(['site_id', 'date'])

# Save unfiltered data before any filtering
unfiltered_df = df.copy()

# Create subsets by filter
unfiltered_by_site = df.groupby('site_id')['date'].apply(list).to_dict()

df = df[df['n_pixels'] >= 5].copy()
df = df.groupby('site_id').filter(lambda x: len(x) >= 20)

useable_by_site = df.groupby('site_id')['date'].apply(list).to_dict()

# Calculate error metrics
groups = df.groupby('site_id')
df = df.assign(
    wse_gauge_norm = df['wse_gauge'] - groups['wse_gauge'].transform('median'),
    wse_swot_norm  = df['wse_swot'] - groups['wse_swot'].transform('median')
)

df['wse_bias'] = df['wse_swot'] - df['wse_gauge']
df['wse_diff'] = df['wse_swot_norm'] - df['wse_gauge_norm']
df['wse_diff_abs'] = df['wse_diff'].abs()

df = df[df['wse_diff_abs'] < 5].copy()

height_by_site = df.groupby('site_id')['date'].apply(list).to_dict()

# Calculate correlation per site
df_for_corr = df.groupby('site_id')[['wse_gauge_norm', 'wse_swot_norm']].corr()
df_for_corr = df_for_corr.unstack().iloc[:, 1]
df['correlation'] = df['site_id'].map(df_for_corr)

# Calculate median error per site
median_error = df.groupby('site_id')['wse_diff_abs'].median()
df['median_wse_diff_abs'] = df['site_id'].map(median_error)

# Filter for high quality
high_qual_df = df[
    (df['correlation'] > 0.6) & 
    (np.abs(df['median_wse_diff_abs']) < 0.3)
].copy()

high_qual_by_site = high_qual_df.groupby('site_id')['date'].apply(list).to_dict()

In [ ]:
# Build comprehensive quality dataframe
quality_records = []

# Start with unfiltered observations
for site_id, dates in unfiltered_by_site.items():
    for date in dates:
        quality_records.append({
            'site_id': site_id,
            'date': date,
            'quality_level': 'unfiltered'
        })

# Overwrite with higher quality levels
for site_id, dates in useable_by_site.items():
    for date in dates:
        quality_records.append({
            'site_id': site_id,
            'date': date,
            'quality_level': 'useable'
        })

for site_id, dates in height_by_site.items():
    for date in dates:
        quality_records.append({
            'site_id': site_id,
            'date': date,
            'quality_level': 'height_threshold'
        })

for site_id, dates in high_qual_by_site.items():
    for date in dates:
        quality_records.append({
            'site_id': site_id,
            'date': date,
            'quality_level': 'high_quality'
        })

# Create dataframe
swot_quality_df = pd.DataFrame(quality_records)

# Keep only the HIGHEST quality level each observation achieved
# (i.e., if it's in high_quality, remove the unfiltered/useable/height versions)
swot_quality_df = swot_quality_df.sort_values(
    ['site_id', 'date', 'quality_level'],
    key=lambda x: x.map({
        'unfiltered': 0,
        'useable': 1,
        'height_threshold': 2,
        'high_quality': 3
    }) if x.name == 'quality_level' else x
)
swot_quality_df = swot_quality_df.drop_duplicates(['site_id', 'date'], keep='last')

# Add the actual stage values from original data
swot_quality_df = swot_quality_df.merge(
    unfiltered_df[['site_id', 'date', 'wse_swot', 'wse_gauge', 'WidthM', 'Bin']],
    on=['site_id', 'date'],
    how='left'
)

# Save
swot_quality_df.to_parquet(os.path.join(save_dir, 'swot_observations_with_quality.parquet'))